<div align="right" style=" font-size: 80%; text-align: center; margin: 0 auto">
<img src="https://raw.githubusercontent.com/Explore-AI/Pictures/master/ExploreAI_logos/Logo blue_dark.png"  style="width:25px" align="right";/>
</div>

<div align="right" style=" font-size: 80%; text-align: center; margin: 0 auto">
<img
 src="https://raw.githubusercontent.com/Explore-AI/Pictures/master/alx-courses/aice/assets/Content_page_banner_blue_dots.png"
 alt="ALX Content Header"
 class="full-width-image"
/>
</div>

# SQL window functions

In this exercise, we will test our understanding and application of SQL window functions on a sample SQLite database file for a retail company called Northwind by performing complex calculations and analyses like ranking, running totals, and date differences. Ensure that you have downloaded the database file, Northwind.db.

## Learning objectives

By the end of this train, you should:
- Use the RANK() function to assign a ranking number to each row based on the order specified within the window.
- Use aggregate window functions to calculate running totals. 
- Use the LAG() function to help calculate the difference, in days, between consecutive date readings in our dataset.
- Use aggregate window functions to calculate the moving average. 

First, let's load our sample database:

In [1]:
!pip install ipython-sql sqlalchemy prettytable

In [1]:
# Load and activate the SQL extension to allow us to execute SQL in a Jupyter notebook.
%load_ext sql


In [2]:
import os
print(os.getcwd())

C:\Users\chris\ALX-data science\data analysis\querying data


In [3]:
# Load the Northwind database stored in your local machine. 
# Make sure the file is saved in the same folder as this notebook.
%sql sqlite:///Northwind.db
    

In [4]:
%%sql
SELECT name FROM sqlite_master WHERE type='table';

 * sqlite:///Northwind.db
Done.


name
Categories
Customers
Region
Shippers
Suppliers
Employees
EmployeeTerritories
OrderDetails
Orders
Territories


Here is a view of all of our tables in the database:

<div align="center" style=" font-size: 80%; text-align: center; margin: 0 auto">
<img src="https://raw.githubusercontent.com/Explore-AI/Pictures/master/Northwind_ERD.png"  style="width:500px";/>
<br>
<br>
    <em>Figure 1: Northwind ERD</em>
</div>

## Exercise

Run the necessary queries that will provide us with the following information. Compare your queries with the solutions at the end of this notebook.

### Exercise 1

Rank all the orders of a specific customer from the most recent to the least recent using window functions. Assume that the customer ID is `'ALFKI'`.

In [11]:
%%sql 

select
    CustomerID,
    OrderDate,
    OrderID,
    dense_rank() over (partition by CustomerID order by OrderDate desc) as order_rank
from 
    Orders
where 
    CustomerID = "ALFKI"
;

 * sqlite:///Northwind.db
Done.


CustomerID,OrderDate,OrderID,order_rank
ALFKI,1998-04-09 00:00:00,11011,1
ALFKI,1998-03-16 00:00:00,10952,2
ALFKI,1998-01-15 00:00:00,10835,3
ALFKI,1997-10-13 00:00:00,10702,4
ALFKI,1997-10-03 00:00:00,10692,5
ALFKI,1997-08-25 00:00:00,10643,6


### Exercise 2

Calculate a running total of the quantity of orders using window functions.

In [17]:
%%sql 

select
    ProductID,
    Quantity,
    OrderID,
    sum(Quantity) over (partition by ProductID order by OrderID ) as orders_running_total
from 
    OrderDetails
;

 * sqlite:///Northwind.db
Done.


ProductID,Quantity,OrderID,orders_running_total
1,45,10285,45
1,18,10294,63
1,20,10317,83
1,15,10348,98
1,12,10354,110
1,15,10370,125
1,10,10406,135
1,24,10413,159
1,15,10477,174
1,40,10522,214


### Exercise 3


Use window functions to find the difference in successive order dates for each customer. **HINT:** The `TIMESTAMPDIFF()` function in MySQL is not available in SQLite. We can use the `julianday()` function to convert the dates to a floating point number and then calculate the difference.

In [23]:
%%sql 

select
    CustomerID,
    OrderDate,
    OrderID,
    Lead(OrderDate) over (partition by CustomerID order by OrderDate ASC) as next_order_date,
    julianday(Lead(OrderDate) over (partition by CustomerID order by OrderDate ASC)) - julianday(OrderDate)  as days_between
from 
    Orders
;

 * sqlite:///Northwind.db
Done.


CustomerID,OrderDate,OrderID,next_order_date,days_between
ALFKI,1997-08-25 00:00:00,10643,1997-10-03 00:00:00,39.0
ALFKI,1997-10-03 00:00:00,10692,1997-10-13 00:00:00,10.0
ALFKI,1997-10-13 00:00:00,10702,1998-01-15 00:00:00,94.0
ALFKI,1998-01-15 00:00:00,10835,1998-03-16 00:00:00,60.0
ALFKI,1998-03-16 00:00:00,10952,1998-04-09 00:00:00,24.0
ALFKI,1998-04-09 00:00:00,11011,None,None
ANATR,1996-09-18 00:00:00,10308,1997-08-08 00:00:00,324.0
ANATR,1997-08-08 00:00:00,10625,1997-11-28 00:00:00,112.0
ANATR,1997-11-28 00:00:00,10759,1998-03-04 00:00:00,96.0
ANATR,1998-03-04 00:00:00,10926,None,None


### Exercise 4

Calculate the moving average of the quantity of the last 3 orders for each product using window functions.

In [40]:
%%sql

SELECT
    OrderID,
    ProductID,
    Quantity,
    AVG(Quantity) OVER (
        PARTITION BY ProductID
        ORDER BY OrderID
        ROWS BETWEEN 2 PRECEDING AND CURRENT ROW
    ) AS MovingAvg_Last3
FROM OrderDetails
ORDER BY ProductID, OrderID;

 * sqlite:///Northwind.db
Done.


OrderID,ProductID,Quantity,MovingAvg_Last3
10285,1,45,45.0
10294,1,18,31.5
10317,1,20,27.666666666666668
10348,1,15,17.666666666666668
10354,1,12,15.666666666666666
10370,1,15,14.0
10406,1,10,12.333333333333334
10413,1,24,16.333333333333332
10477,1,15,16.333333333333332
10522,1,40,26.333333333333332


In [34]:
%%sql

SELECT
    ProductID,
    Quantity,
    OrderID,
    AVG(Quantity) OVER (
        PARTITION BY ProductID
        ORDER BY OrderID DESC
        ROWS BETWEEN CURRENT ROW AND 2 FOLLOWING
    ) AS moving_avg_last_3
FROM OrderDetails;

 * sqlite:///Northwind.db
Done.


ProductID,Quantity,OrderID,moving_avg_last_3
1,40,11070,25.0
1,25,11047,26.666666666666668
1,10,11035,21.666666666666668
1,45,11031,21.0
1,10,11025,6.666666666666667
1,8,11006,4.666666666666667
1,2,11005,9.0
1,4,11003,28.333333333333332
1,21,10935,30.333333333333332
1,60,10918,30.0


In [38]:
%%sql

SELECT
    ProductID,
    Quantity,
    OrderID,
    order_number,
    AVG(Quantity) OVER (
        PARTITION BY ProductID
        ORDER BY OrderID DESC
        ROWS BETWEEN CURRENT ROW AND 2 FOLLOWING
    ) AS moving_avg_last_3
FROM (
    SELECT
        ProductID,
        Quantity,
        OrderID,
        ROW_NUMBER() OVER (PARTITION BY ProductID ORDER BY OrderID DESC) AS order_number
    FROM OrderDetails
) AS numbered_orders
WHERE order_number <= 3;

 * sqlite:///Northwind.db
Done.


ProductID,Quantity,OrderID,order_number,moving_avg_last_3
1,40,11070,1,25.0
1,25,11047,2,17.5
1,10,11035,3,10.0
2,24,11077,1,14.0
2,10,11075,2,9.0
2,8,11072,3,8.0
3,4,11077,1,19.666666666666668
3,25,11017,2,27.5
3,30,10857,3,30.0
4,1,11077,1,10.333333333333334


In [37]:
%%sql

SELECT 
    OrderID, 
    ProductID, 
    Quantity,
    AVG(Quantity) OVER (
        PARTITION BY ProductID
        ORDER BY OrderID DESC
        ROWS BETWEEN CURRENT ROW AND 2 FOLLOWING
    ) AS MovingAvgQuantity
FROM 
    OrderDetails
ORDER BY 
    ProductID, 
    OrderID DESC;

 * sqlite:///Northwind.db
Done.


OrderID,ProductID,Quantity,MovingAvgQuantity
11070,1,40,25.0
11047,1,25,26.666666666666668
11035,1,10,21.666666666666668
11031,1,45,21.0
11025,1,10,6.666666666666667
11006,1,8,4.666666666666667
11005,1,2,9.0
11003,1,4,28.333333333333332
10935,1,21,30.333333333333332
10918,1,60,30.0


## Solutions

### Exercise 1

In [ ]:
%%sql

SELECT 
    OrderID, 
    OrderDate,
    RANK() OVER (ORDER BY OrderDate DESC) as Order_rank
FROM 
    Orders 
WHERE 
    CustomerID = 'ALFKI';

The `RANK()` window function is used here to rank each order of the customer with the ID `'ALFKI'` based on the `OrderDate`. The `DESC` keyword is used so that the most recent order gets the highest rank (i.e. 1).

### Exercise 2

In [14]:
%%sql

SELECT 
    OrderID, 
    Quantity, 
    SUM(Quantity) OVER (
    ORDER BY OrderID) as RunningTotal 
FROM 
    OrderDetails
GROUP BY 
    OrderID;


 * sqlite:///Northwind.db
Done.


OrderID,Quantity,RunningTotal
10248,12,12
10249,9,21
10250,10,31
10251,6,37
10252,40,77
10253,20,97
10254,15,112
10255,20,132
10256,15,147
10257,25,172


The `SUM()` window function is used here to calculate a running total of the quantity of orders. The `ORDER BY` clause inside the `OVER()` clause ensures that the running total is calculated in the order of the `OrderID`. We then group our data by `OrderID`.

### Exercise 3

In [ ]:
%%sql

SELECT 
    CustomerID, 
    OrderDate, 
    LAG(OrderDate, 1) OVER 
        (PARTITION BY CustomerID 
        ORDER BY OrderDate) as PrevOrderDate, 
    julianday(OrderDate)-
    julianday(LAG(OrderDate, 1) OVER
            (PARTITION BY CustomerID 
            ORDER BY OrderDate)) as DateDiff
FROM 
    Orders;

The `LAG()` window function is used twice here, once to get the previous order date for each customer, and again to calculate the difference between the current order date and the previous order date. The `PARTITION BY` clause is used to separate the data into partitions based on the `CustomerID`. 

Since the `TIMESTAMPDIFF()` function in MySQL is not available in SQLite, we use the `julianday()` function to convert the dates to a floating point number, which we can then use for subtraction to find the difference between two dates.

### Exercise 4

In [35]:
%%sql

SELECT 
    OrderID, 
    ProductID, 
    Quantity,
    AVG(Quantity) OVER (PARTITION BY ProductID ORDER BY OrderID ROWS BETWEEN 2 PRECEDING AND 0 FOLLOWING) as MovingAvgQuantity
FROM 
    OrderDetails
ORDER BY 
    ProductID, 
    OrderID;

 * sqlite:///Northwind.db
Done.


OrderID,ProductID,Quantity,MovingAvgQuantity
10285,1,45,45.0
10294,1,18,31.5
10317,1,20,27.666666666666668
10348,1,15,17.666666666666668
10354,1,12,15.666666666666666
10370,1,15,14.0
10406,1,10,12.333333333333334
10413,1,24,16.333333333333332
10477,1,15,16.333333333333332
10522,1,40,26.333333333333332


The `AVG()` window function is used here to calculate the moving average of `Quantity` for the last 3 orders (the current order and the two preceding orders) for each product. The window is defined using the `PARTITION BY` clause (to segment the data by `ProductID`) and the `ORDER BY` clause (to arrange the data in order of `OrderID`). The `ROWS BETWEEN` clause specifies the size and location of the window – in this case, **the current row and the two rows preceding it.**

#

<div align="center" style=" font-size: 80%; text-align: center; margin: 0 auto">
<img src="https://raw.githubusercontent.com/Explore-AI/Pictures/refs/heads/master/ALX_banners/ALX_Navy.png"  style="width:100px"  ;/>
</div>